In [11]:
! pip install retry_requests

  Using cached retry_requests-2.0.0-py3-none-any.whl.metadata (2.6 kB)
Using cached retry_requests-2.0.0-py3-none-any.whl (15 kB)


In [12]:
import pandas as pd
import os
import requests
import urllib.robotparser as rp
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import random as rand
import pandas as pd
import re
import time
import numpy as np
from geopy.geocoders import Nominatim
import openmeteo_requests

import pandas as pd
from retry_requests import retry
# import requests_cache


In [15]:
extracted_data_output_path = os.path.join(os.getcwd(), '..', 'data', 'data_extracted')

In [13]:
stadiums = {
    'Arsenal': 'Emirates Stadium',
    'Aston Villa': 'Villa Park',
    'Bournemouth': 'Vitality Stadium',
    'Brentford': 'Gtech Community Stadium',
    'Brighton': 'American Express Community Stadium',
    'Burnley': 'Turf Moor',
    'Cardiff': 'Cardiff City Stadium',
    'Chelsea': 'Stamford Bridge',
    'Crystal Palace': 'Selhurst Park',
    'Everton': 'Goodison Park',
    'Fulham': 'Craven Cottage',
    'Huddersfield': 'John Smith\'s Stadium',
    'Ipswich': 'Portman Road Satdium',
    'Leicester': 'King Power Stadium',
    'Leeds': 'Elland Road Stadium',
    'Liverpool': 'Anfield',
    'Luton': "Kenilworth Road Stadium",
    'Manchester City': "Etihad Stadium",
    'Manchester United': "Old Trafford",
    'Newcastle': "St James' Park",
    'Norwich': "Carrow Road Stadium",
    'Nottingham Forest': "City Ground",
    'Sheffield United': "Bramall Lane",
    'Southampton': "St Mary's Stadium",
    'Stoke': "Bet365 Stadium",
    'Swansea': "Swansea.com Stadium",
    'Tottenham': "Tottenham Hotspur Stadium",
    'Tottenham Old': "Wembley Stadium",
    'Watford': "Vicarage Road Stadium",
    'West Ham': "London Stadium",
    'West Brom': "The Hawthorns",
    "Wolves": "Molineux Stadium",
}


In [ ]:

geolocator = Nominatim(user_agent="GetLoc", timeout=10)
stadium_loc_df = pd.DataFrame(columns=['team', 'stadium', 'latitude', 'longitude'])
for team, stadium in stadiums.items():
    location = geolocator.geocode(stadium + ', UK')
    time.sleep(2)  # To avoid hitting the rate limit

    if location is None:
        print(f"Could not find location for {stadium} in {team}.")
        continue
    stadium_loc_df.loc[len(stadium_loc_df)] = [
        team,
        stadium,
        location.latitude,
        location.longitude
    ]
    
stadium_loc_df.loc[len(stadium_loc_df)] = [
    'Sheffield United',
    'Bramall Lane',
    53.369871,
    -1.469003
]
stadium_loc_df.loc[len(stadium_loc_df)] = [
    'Ipswich',
    'Portman Road Satdium',
    52.054509,
    1.146325
]
stadium_loc_df.to_csv(os.path.join(extracted_data_output_path, 'stadiums_location.csv'), index=False)
stadium_loc_df
    

Could not find location for Portman Road Satdium in Ipswich.
Could not find location for Bramall Lanen Stadium in Sheffield United.


,team,stadium,latitude,longitude
0,Arsenal,Emirates Stadium,51.555040,-0.108400
1,Aston Villa,Villa Park,52.509126,-1.885035
2,Bournemouth,Vitality Stadium,50.735181,-1.838328
3,Brentford,Gtech Community Stadium,51.490713,-0.288980
4,Brighton,American Express Community Stadium,50.861547,-0.083693
5,Burnley,Turf Moor,53.789361,-2.229855
6,Cardiff,Cardiff City Stadium,51.472821,-3.202963
7,Chelsea,Stamford Bridge,51.481687,-0.191034
8,Crystal Palace,Selhurst Park,51.398243,-0.085255
9,Everton,Goodison Park,53.438690,-2.966419


In [ ]:
min_data = '2017-08-11'
max_data = '2025-05-26'

weather_data = None

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)
url = "https://archive-api.open-meteo.com/v1/archive"

for index, row in stadium_loc_df.iterrows():
    
	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	params = {
		"latitude": row['latitude'],
		"longitude": row['longitude'],
		"start_date": min_data,
		"end_date": max_data,
		"hourly": ["temperature_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "surface_pressure", "wind_speed_10m", "relative_humidity_2m", "wind_gusts_10m"],
		"timezone": "auto"
	}
	responses = openmeteo.weather_api(url, params=params)

	# Process first location. Add a for-loop for multiple locations or weather models
	response = responses[0]
	print(f"Stadium: {row['stadium']}. Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation {response.Elevation()} m asl")
	print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
	print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
	hourly_apparent_temperature = hourly.Variables(1).ValuesAsNumpy()
	hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
	hourly_rain = hourly.Variables(3).ValuesAsNumpy()
	hourly_snowfall = hourly.Variables(4).ValuesAsNumpy()
	hourly_surface_pressure = hourly.Variables(5).ValuesAsNumpy()
	hourly_wind_speed_10m = hourly.Variables(6).ValuesAsNumpy()
	hourly_rel_humidity_2m = hourly.Variables(7).ValuesAsNumpy()
	hourly_wind_gusts_10m = hourly.Variables(8).ValuesAsNumpy()


	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}

	hourly_data["temperature_2m"] = hourly_temperature_2m
	hourly_data["apparent_temperature"] = hourly_apparent_temperature
	hourly_data["precipitation"] = hourly_precipitation
	hourly_data["rain"] = hourly_rain
	hourly_data["snowfall"] = hourly_snowfall
	hourly_data["surface_pressure"] = hourly_surface_pressure
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["relative_humidity_2m"] = hourly_rel_humidity_2m
	hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m

	hourly_dataframe = pd.DataFrame(data = hourly_data)
	hourly_dataframe['team'] = row['team']
	hourly_dataframe['stadium'] = row['stadium']
 	hourly_dataframe['latitude'] = row['latitude']
	hourly_dataframe['longitude'] = row['longitude']
	if weather_data is None:
		weather_data = hourly_dataframe
	else:
		weather_data = pd.concat([weather_data, hourly_dataframe], ignore_index=True)
	
	time.sleep(30)  # To avoid hitting the rate limit


weather_data

Stadium: The Hawthorns. Coordinates: 52.478031158447266°N -2.0074462890625°E
Elevation 171.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Molineux Stadium. Coordinates: 52.61862564086914°N -2.182861328125°E
Elevation 135.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Bramall Lane. Coordinates: 53.39191436767578°N -1.371429443359375°E
Elevation 74.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s
Stadium: Portman Road Satdium. Coordinates: 52.056236267089844°N 1.158088207244873°E
Elevation 3.0 m asl
Timezone b'Europe/London'b'GMT+1'
Timezone difference to GMT+0 3600 s


,date,temperature_2m,apparent_temperature,precipitation,rain,snowfall,surface_pressure,wind_speed_10m,relative_humidity_2m,wind_gusts_10m,team,stadium
0,2017-08-10 23:00:00+00:00,10.365499,9.503113,0.0,0.0,0.0,1001.785278,5.091168,94.469589,10.799999,West Brom,The Hawthorns
1,2017-08-11 00:00:00+00:00,10.165500,9.087279,0.0,0.0,0.0,1001.281067,6.287130,94.779236,8.640000,West Brom,The Hawthorns
2,2017-08-11 01:00:00+00:00,9.815499,8.517171,0.0,0.0,0.0,1000.765686,7.421590,95.727852,13.320000,West Brom,The Hawthorns
3,2017-08-11 02:00:00+00:00,8.965500,7.747960,0.0,0.0,0.0,1000.213989,5.411986,96.024490,12.959999,West Brom,The Hawthorns
4,2017-08-11 03:00:00+00:00,7.965500,6.592342,0.0,0.0,0.0,999.846802,5.091168,97.312859,9.360000,West Brom,The Hawthorns
...,...,...,...,...,...,...,...,...,...,...,...,...
273211,2025-05-26 18:00:00+00:00,15.019000,12.691244,0.1,0.1,0.0,1013.139709,15.048042,69.430687,44.639996,Ipswich,Portman Road Satdium
273212,2025-05-26 19:00:00+00:00,14.568999,12.178141,0.0,0.0,0.0,1013.138977,16.055355,72.926758,30.599998,Ipswich,Portman Road Satdium
273213,2025-05-26 20:00:00+00:00,14.169000,11.679056,0.1,0.1,0.0,1012.838501,17.123669,75.846313,32.399998,Ipswich,Portman Road Satdium
273214,2025-05-26 21:00:00+00:00,14.068999,10.955987,0.0,0.0,0.0,1012.338684,20.719555,74.569618,39.959999,Ipswich,Portman Road Satdium


In [ ]:
weather_data.to_csv(os.path.join(extracted_data_output_path, 'stadiums_weather_to_the_end.csv'), index=False)